In [80]:
import pandas as pd
import spacy
from collections import defaultdict 
from spacy import displacy
import random
from spacy.training import Example
from typing import List, Tuple
from sentence_transformers import SentenceTransformer
import nltk
from nltk.corpus import stopwords
import string
import re
import numpy as np
from tqdm import tqdm

# 1. Загрузка датасета

In [82]:
df = pd.read_json("../data/data.txt", lines=True)
final_df = df.copy()

# 2. Используем стандартную NER-модель

In [83]:
# Загружаем модель для русского языка
# если она не найдена, то установим командой: python -m spacy download ru_core_news_lg
# также попробовать deeppavlovru_core_news_lg - https://chat.deepseek.com/a/chat/s/3d0a9423-ea38-4e29-8dc5-63b5ec94b45a
nlp = spacy.load("ru_core_news_lg")

In [84]:
def extract_entities_from_texts(texts):
    results = []
    for doc in nlp.pipe(texts, batch_size=50, n_process=-1):  # n_process=-1 использует все ядра
        entities = defaultdict(set)
        for ent in doc.ents:
            entities[ent.label_].add(ent.text)
        results.append(dict(entities))
    return results

final_df['entities'] = extract_entities_from_texts(final_df['text'].str.lower().tolist())

In [91]:
final_df



,text,tags,schema_name,table_name,entities
0,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47923,"{'LOC': {'москвы', 'рф'}, 'ORG': {'минприроды'}}"
1,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,"{'LOC': {'москвы', 'российской федерации'}}"
2,"2 2 2 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,{'LOC': {'москвы'}}
3,"3 3 3 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,{'LOC': {'москвы'}}
4,"4 4 4 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,{'LOC': {'москвы'}}
...,...,...,...,...,...
18419,996 996 982 20231117_110117.jpg 2023-11-17 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,{}
18420,997 997 983 20231121_100442.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,{'PER': {'пип битцевский'}}
18421,998 998 984 20231121_095847.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,{'PER': {'пип битцевский'}}
18422,99 99 101 117 4.jpg 2023-11-16 00:00:00+00 13:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,{}


In [92]:
# визуализация
# text = final_df[5:6].reset_index()['text'].str.lower()[0]
text = final_df[100:101].reset_index()['text'][0]
doc = nlp(text)
displacy.render(doc, style="ent", jupyter=True)

# 3. Добавим labels и дообучим существующую NER-модель

In [93]:
# СЗАО - Северо-Западный административный округ
# САО - Северный административный округ
# СВАО - Северо-Восточный административный округ
# ЗАО - Западный административный округ
# ЦАО - Центральный административный округ
# ВАО - Восточный административный округ
# ЮЗАО - Юго-Западный административный округ
# ЮАО - Южный административный округ
# ЮВАО - Юго-Восточный административный округ
# ЗелАО - Зеленоградский административный округ
# ТиНАО - Троицкий и Новомосковский административные округа
# НАО - Новомосковский административный округ
# ТАО - Троицкий административный округ

In [94]:
result_df = df.copy()
moscow_districts = ['СЗАО', 'САО', 'СВАО', 'ЗАО', 'ЦАО', 'ВАО', 'ЮЗАО', 'ЮАО', 'ЮВАО', 'ЗелАО', 'ТиНАО', 'НАО', 'ТАО']
# Создаем регулярное выражение для поиска любого из значений
pattern = '|'.join(moscow_districts)
result_df = df[df['text'].str.contains(pattern, case=False, na=False)]

In [95]:
result_df

,text,tags,schema_name,table_name
72,"100 101 64 ООПТ регионального значения ""Памятн...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_54935
75,"102 103 3 ООПТ регионального значения ""Памятни...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_54935
118,"141 142 120 ООПТ регионального значения ""Фауни...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_54935
144,"31 31 138 ООПТ регионального значения ""Памятни...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_54935
215,"96 97 38 ООПТ регионального значения ""Природно...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_54935
...,...,...,...,...
18419,996 996 982 20231117_110117.jpg 2023-11-17 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18420,997 997 983 20231121_100442.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18421,998 998 984 20231121_095847.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18422,99 99 101 117 4.jpg 2023-11-16 00:00:00+00 13:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876


In [96]:
def find_all_positions(text, substring):
    """Находит все позиции вхождения подстроки в тексте"""
    positions = []
    start = 0
    while True:
        pos = text.find(substring, start)
        if pos == -1:
            break
        positions.append([pos, pos + len(substring)-1])
        start = pos + len(substring) # ищем следующее вхождение
    return positions
# Пример использования
text = "СЗАО и ВАО - это округа Москвы. СЗАО находится на северо-западе. СЗАО" 
substring = "СЗАО"
positions = find_all_positions(text, substring)
print(f"Подстрока '{substring}' найдена на позициях: {positions}")

Подстрока 'СЗАО' найдена на позициях: [[0, 3], [32, 35], [65, 68]]


In [ ]:
def 

In [97]:
def get_examples(df, entities_dict):
    all_annotated_examples = []
    texts = df['text'].tolist()
    for text in texts:
        example_for_text = []
        entities_for_text = []
        for entity in entities_dict:
            for value in entities_dict[entity]:
                # nonlocal positions
                positions = find_all_positions(text, value)
                for position in positions:
                    position.append(entity)
                    entities_for_text.append(position)
        
        if len(entities_for_text) > 0:
            example_for_text.insert(0, text)
            example_for_text.append({'entities': entities_for_text})
            all_annotated_examples.append(example_for_text)
    return all_annotated_examples

entities_values = {"DISTRICT":moscow_districts}
annotated_examples = get_examples(result_df[:100], entities_values)
print(annotated_examples)
        

[['100 101 64 ООПТ регионального значения "Памятник природы "Долина реки Чермянки от пр. Дежнева до устья" ППМ № 2119-ПП от 18.09.2024,ППМ № 1496-ПП от 11.09.2020 Памятники природы Утвержден ГБУ г. Москвы "Автомобильные дороги СВАО" https://docs7.online-sps.ru/cgi/online.cgi?from=228884-0&req=doc&rnd=JMzqYA&base=MLAW&n=246251#mUvXRTUQQIa3BB3w https://www.mos.ru/upload/content/files/020KDPPDolinarekiChermyankiotprDejnevadoystya(2).docx 15.55 15.5477 не совпадает с зонами режимов ООПТ (по координатам в ППМ - такая же геометрия) Иль С.А.: Проверен Мукаяров Е.А.: Внесено в соответствии с ППМ 2119 от 18.09.24', {'entities': [[224, 227, 'DISTRICT'], [225, 227, 'DISTRICT']]}], ['102 103 3 ООПТ регионального значения "Памятник природы "Пойма реки Городни от Братеевской ул. до реки Москвы" ППМ № 2406-ПП от 23.10.2024,ППМ № 1540-ПП от 16.09.2020 Памятники природы Утвержден ГБУ г. Москвы "Автомобильные дороги ЮАО" https://docs7.online-sps.ru/cgi/online.cgi?req=doc&base=MLAW&n=247272&cacheid=2983D

In [106]:
def resolve_overlapping_entities(training_sample):
    """
    Гарантированно оставляет наибольшие интервалы из пересекающихся сущностей.
    """
    text, annotations = training_sample
    entities = annotations.get('entities', [])
    
    if not entities:
        return training_sample
    
    # Удаляем точные дубликаты
    unique_entities = list(set(tuple(entity) for entity in entities))
    
    # Группируем пересекающиеся сущности и выбираем наибольшую из каждой группы
    result_entities = []
    
    # Сортируем по начальной позиции для группировки пересечений
    sorted_entities = sorted(unique_entities, key=lambda x: x[0])
    
    i = 0
    n = len(sorted_entities)
    
    while i < n:
        current_entity = sorted_entities[i]
        current_start, current_end, current_label = current_entity
        best_entity = current_entity
        best_length = current_end - current_start
        
        # Ищем все сущности, пересекающиеся с текущей
        j = i + 1
        while j < n:
            next_entity = sorted_entities[j]
            next_start, next_end, next_label = next_entity
            
            # Если сущности пересекаются
            if next_start < current_end:
                next_length = next_end - next_start
                # Выбираем сущность с наибольшей длиной
                if next_length > best_length:
                    best_entity = next_entity
                    best_length = next_length
                j += 1
            else:
                break
        
        # Добавляем лучшую сущность из группы пересекающихся
        result_entities.append(list(best_entity))
        
        # Пропускаем все обработанные пересекающиеся сущности
        i = j
    
    # Сортируем результат по начальной позиции
    result_entities.sort(key=lambda x: x[0])
    
    return [text, {'entities': result_entities}]

In [107]:
resolved_annotated_examples = []
for example in annotated_examples:
    resolved_example = resolve_overlapping_entities(example)
    resolved_annotated_examples.append(resolved_example)
print(resolved_annotated_examples)

[['99 100 17 ООПТ регионального значения "Памятник природы "Родник на левобережном склоне долины реки Яузы в Старом Свиблово" ППМ № 2119 от 18.09.2024,ППМ № 2809-ПП от 13.12.2022,ППМ № 1496-ПП от 11.09.2020 Памятники природы Утвержден ГБУ г. Москвы "Автомобильные дороги СВАО" https://docs7.online-sps.ru/cgi/online.cgi?req=doc&base=MLAW&n=246251&cacheid=3187E19D12CA3284889FF10C119827F3&mode=splus&rnd=svztFw#NewERTU6F9Il4eeD1 https://www.mos.ru/upload/content/files/050KDPPRodniknalevoberejnomsklonedolinirekiYayzivStaromSviblovo.docx 0.6 0.6015 проверено Иль С.А.: Проверен Мукаяров Е.А.: Внесено в соответствии с ППМ 2119 от 18.09.24', {'entities': [[268, 271, 'DISTRICT']]}], ['108 111 Памятник природы "Родники в долине реки Химки в природно-историческом парке "Покровское-Стрешнево" 053 1991-10-17 00:00:00+00 Памятник природы «Родники в долине реки Химки в природно-историческом парке «Покровское-Стрешнево» образован с целью охраны гидрологического объекта, а также прилегающих к нему природ

In [112]:
# Получаем NER компонент
ner = nlp.get_pipe("ner")

In [113]:
# добавляем новые labels
for label in entities_values:
    if label not in ner.labels:
        ner.add_label(label)
        print(f"Добавлен новый label: {label}")

In [114]:
# отключаем все остальные pipe
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]
with nlp.disable_pipes(*other_pipes):
    optimizer = nlp.resume_training()

In [115]:
# 
iteration_number = 5
for iteration in range(iteration_number):
    random.shuffle(resolved_annotated_examples)
    losses = {}
    for example in resolved_annotated_examples:
        text = example[0]
        annotations = example[1]
        doc = nlp.make_doc(text)
        train_example = Example.from_dict(doc, annotations)
        nlp.update([train_example], sgd=optimizer, losses=losses, drop=0.4)
    if iteration % 5 == 0:
        print(f"Iteration {iteration}, Loss: {losses['ner']}")


D:\Artem\Magistrature\1_sem\intellg_systems_and_techno\labs\.venv\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "135 138 Памятник природы "Четыре родника в Голосов..." with entities "[[828, 830, 'DISTRICT'], [851, 853, 'DISTRICT'], [...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
D:\Artem\Magistrature\1_sem\intellg_systems_and_techno\labs\.venv\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "124 127 Памятник природы "Родник на левом берегу р..." with entities "[[1148, 1152, 'DISTRICT'], [1181, 1185, 'DISTRICT'...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
D:\Artem\Magistrature\1_sem\inte

KeyboardInterrupt: 

In [ ]:
ner.to_disk('../files')